In [1]:
import sys
#sys.path.append('/Users/theodorehuppert/VSCode/pyNIRS_toolbox/pyNIRS_toolbox')

import pyBrainAnalyzIR
import pandas as pd
import pyBrainAnalyzIR.testing

# All the processing modules are in pipelines.modules
import pyBrainAnalyzIR
import pyBrainAnalyzIR.pipelines.modules as pipelines
import pyBrainAnalyzIR.dataclasses.dataset as dataset



In [2]:
dset=dataset.DataSet()
data1,_=pyBrainAnalyzIR.testing.simData.Data(snr=5)
data2,_=pyBrainAnalyzIR.testing.simData.Data(snr=5)
data3,_=pyBrainAnalyzIR.testing.simData.Data(snr=5)
data4,_=pyBrainAnalyzIR.testing.simData.Data(snr=5)
data5,_=pyBrainAnalyzIR.testing.simData.Data(snr=5)
data6,_=pyBrainAnalyzIR.testing.simData.Data(snr=5)

dset.import_data(data1)
dset.import_data(data2)
dset.import_data(data3)
dset.import_data(data4)
dset.import_data(data5)
dset.import_data(data6)

demo=pd.DataFrame({'subject':['A','B','C','D','E','F'],'gender':['M','M','F','F','M','F'],'age':[1.,3.,5.,6.,1.,3.]})
dset.add_demographics_by_index(demo)

print("\n\nRunning Analysis:\n")
job = pipelines.events.rename_stims()
job.options['ListofChanges']={
        "1.0": "control",
        "2.0": "Tapping/Left",
        "3.0": "Tapping/Right",
        "15.0": "start marker",
    }
job = pipelines.events.remove_stims(job)
job.options['ListtoRemove']=["start marker"]
job = pipelines.preproccessing.intensity_opticaldensity(job)
#job = pipelines.filters.bandpass_filter(job)
job = pipelines.preproccessing.mbll(job)
job = pipelines.preproccessing.resample(job)
job.options['Fs']=1
job = pipelines.motion_correction.TDDR(job)
job = pipelines.glm.GLM(job)
job.options['noise_model']='ar_irls'


dset=job.run(dset)


 94%|█████████▍| 15/16 [00:00<00:00, 865.75it/s]




Running Analysis:



100%|██████████| 16/16 [00:00<00:00, 34.53it/s]


In [3]:
job = pipelines.mixedeffects.MixedEffects()
job.options['FE_formula']='Beta ~ 0 + Condition+Condition:age'
job.options['robust']=True

dset=job.run(dset)

LinAlgError: Matrix is not positive definite

In [ ]:
dset['groupstats'].table()

,Channel,Type,Condition,Beta,StdErr,T-value,P-values,Q-values
0,S1D1,HbO,Condition[Drift 0],-1.219602,0.374439,-3.257148,1.330631e-03,3.784907e-03
1,S1D1,HbO,Condition[HRF A],0.131934,3.312269,0.039832,9.682686e-01,9.989666e-01
2,S1D1,HbO,Condition[Drift 0]:age,-2.500986,0.231906,-10.784473,1.669544e-21,2.671271e-20
3,S1D1,HbO,Condition[HRF A]:age,0.072181,2.070148,0.034868,9.722215e-01,9.989666e-01
4,S2D1,HbO,Condition[Drift 0],-1.720407,0.408310,-4.213485,3.865823e-05,1.150757e-04
...,...,...,...,...,...,...,...,...
123,S8D8,HbR,Condition[HRF A]:age,0.025486,1.096450,0.023244,9.814799e-01,9.989666e-01
124,S9D8,HbR,Condition[Drift 0],2.383612,0.338947,7.032411,3.470685e-11,1.645362e-10
125,S9D8,HbR,Condition[HRF A],-0.151530,2.517921,-0.060180,9.520745e-01,9.989666e-01
126,S9D8,HbR,Condition[Drift 0]:age,0.312053,0.150986,2.066770,4.009767e-02,9.331822e-02
